# Hangman wordlist explorer

Play with the curated master wordlist and the `wordlist.py` loader.
Run from anywhere in the project (repo root, `math/`, or `math/notebooks/`)
— the setup cell finds `wordlist.py` and imports it.

The list is `data/wordlist_master.txt` (`word<TAB>zipf`, frequency-sorted),
built by `math/build_wordlist.py`. `zipf` is wordfreq's frequency scale:
higher = more common (4.0 ≈ 10/million, 3.0 ≈ 1/million, 2.0 = the file floor).
The loader's default cutoff is `DEFAULT_ZIPF = 2.5`.

In [1]:
from pathlib import Path
import sys

def find_math_dir():
    """Find the directory containing wordlist.py (the math/ dir)."""
    cwd = Path.cwd().resolve()
    for candidate in (cwd, cwd.parent, cwd.parent.parent, cwd / 'math', cwd.parent / 'math'):
        if (candidate / 'wordlist.py').exists():
            return candidate
    raise FileNotFoundError(
        'Could not find wordlist.py. Open this notebook from the project root, '
        'math/, or math/notebooks/.'
    )

math_dir = find_math_dir()
if str(math_dir) not in sys.path:
    sys.path.insert(0, str(math_dir))

import wordlist
from wordlist import load, tier, Word, DEFAULT_ZIPF, TIERS

print(f'imported wordlist from {math_dir}')
print(f'DEFAULT_ZIPF = {DEFAULT_ZIPF}')

imported wordlist from /Users/patrickliscio/My Documents/Pycharm Projects/Ballpark-Figures/hangman/math
DEFAULT_ZIPF = 2.5


## The whole list at a glance

In [2]:
full = wordlist._raw()   # every stored word, frequency order, no cutoff
print(f'{len(full):,} words total, zipf {full[-1].zipf:.2f} .. {full[0].zipf:.2f}')
print(f'{len(load()):,} words at the default cutoff (zipf >= {DEFAULT_ZIPF})')
print()
print('most common 15:', ', '.join(w.text for w in full[:15]))
print('rarest 15:     ', ', '.join(w.text for w in full[-15:]))

47,294 words total, zipf 2.00 .. 7.73
34,483 words at the default cutoff (zipf >= 2.5)

most common 15: the, to, and, of, a, in, i, is, for, that, you, it, on, with, this
rarest 15:      uninviting, upcountry, vacuoles, vend, victuals, voile, voyageurs, wackos, wallah, whetted, whimsically, whoso, wittily, wretchedness, yurts


## Tune the cutoff

Change `CUTOFF` and re-run to see how many words survive and what the
boundary looks like. Words *just above* the cutoff are the hardest ones
still included — a good gut-check for whether the cutoff feels right.

In [3]:
CUTOFF = 2.5   # <-- try 3.0, 2.8, 2.2, 2.0 ...

words = load(min_zipf=CUTOFF)
print(f'zipf >= {CUTOFF}: {len(words):,} words\n')

# 30 words sitting right at the cutoff (the marginal, hardest-kept words)
near = [w for w in words if w.zipf < CUTOFF + 0.06]
print(f'{len(near)} words within 0.05 of the cutoff — sample:')
print(', '.join(w.text for w in near[:30]))

zipf >= 2.5: 34,483 words

1542 words within 0.05 of the cutoff — sample:
abducting, abides, aboriginals, acrimonious, adorning, adverb, anarchic, arras, attache, backtracking, bandy, barnacle, barometric, beguiling, bendy, berets, bevy, bidirectional, blanked, breastfed, bronzed, brunettes, bunched, bureaucracies, bushings, buzzards, calipers, certifies, chaparral, chivalrous


## Filter by length and frequency

`load()` takes `min_len`, `max_len`, `min_zipf`, `max_zipf`. Handy combo:
a `min_zipf`/`max_zipf` *band* isolates "known but not trivially common"
words — good for a harder round.

In [4]:
# e.g. 6-letter words in the 'known but not everyday' band
sel = load(min_len=6, max_len=6, min_zipf=2.5, max_zipf=3.5)
print(f'{len(sel):,} six-letter words, zipf 2.5-3.5')
print(', '.join(w.text for w in sel[:40]))

2,903 six-letter words, zipf 2.5-3.5
ankles, avoids, badges, bridal, buckle, bursts, chilly, chords, clowns, cradle, crappy, fellas, haters, hitter, legged, mailed, mantra, mapped, mating, outing, peaked, primer, router, shovel, splits, toxins, twists, attire, barley, bitten, blasts, braces, bumped, catchy, diaper, erased, hailed, hiatus, ledger, lodged


## Distributions

Text histograms (no plotting deps). How word length and frequency are spread
across the list at the current cutoff — useful for sanity-checking difficulty.

In [5]:
from collections import Counter

def bar(count, scale):
    return '█' * int(count / scale)

words = load(min_zipf=CUTOFF)

print(f'Word length distribution (zipf >= {CUTOFF}):')
by_len = Counter(len(w.text) for w in words)
scale = max(by_len.values()) / 40
for length in range(min(by_len), max(by_len) + 1):
    c = by_len.get(length, 0)
    print(f'  {length:>2} | {bar(c, scale):<40} {c:,}')

print(f'\nWords per 0.5-zipf band:')
by_band = Counter((int(w.zipf * 2) / 2) for w in words)
scale = max(by_band.values()) / 40
for band in sorted(by_band, reverse=True):
    c = by_band[band]
    print(f'  {band:>4.1f} | {bar(c, scale):<40} {c:,}')

Word length distribution (zipf >= 2.5):
   1 |                                          2
   2 |                                          49
   3 | ████                                     640
   4 | ███████████████                          2,120
   5 | █████████████████████████                3,495
   6 | ██████████████████████████████████       4,824
   7 | ████████████████████████████████████████ 5,541
   8 | ██████████████████████████████████████   5,267
   9 | ████████████████████████████████         4,505
  10 | ████████████████████████                 3,359
  11 | ███████████████                          2,142
  12 | █████████                                1,310
  13 | █████                                    709
  14 | ██                                       302
  15 | █                                        147
  16 |                                          47
  17 |                                          18
  18 |                                          3
  19 |       

In [7]:
length = 7
sel = load(min_len=length)
words = [x.text for x in sel]
print(len(words))
print(words)
l = [(let, len([w for w in words if let in w])) for let in 'abcdefghijklmnopqrstuvwxyz']
x = sorted(l, key=lambda x: -x[1])
for i in x:
    print(i)

23353
['because', 'through', 'something', 'between', 'another', 'without', 'against', 'someone', 'company', 'different', 'thought', 'however', 'getting', 'government', 'looking', 'business', 'already', 'anything', 'nothing', 'everything', 'support', 'including', 'believe', 'national', 'actually', 'service', 'country', 'children', 'everyone', 'general', 'following', 'together', 'president', 'working', 'important', 'friends', 'information', 'control', 'problem', 'history', 'research', 'several', 'started', 'university', 'remember', 'playing', 'probably', 'understand', 'international', 'possible', 'members', 'question', 'community', 'special', 'million', 'morning', 'whether', 'development', 'further', 'minutes', 'services', 'yourself', 'although', 'players', 'building', 'political', 'talking', 'according', 'available', 'education', 'sometimes', 'college', 'current', 'example', 'experience', 'program', 'process', 'himself', 'outside', 'students', 'instead', 'thinking', 'department', 'resul

## Letter position patterns

For a chosen word length, tabulate for each letter every subset of positions
it occupies across the list — e.g. length 3, `e`: `___`, `e__`, `_e_`, `__e`,
`ee_`, `e_e`, `_ee`. Columns: `count` (# words) and `share` (fraction of words)
are unweighted; `freq` and `prob` weight each word by how common it is.
Logic lives in `letter_patterns.py`.

In [8]:
import letter_patterns

LENGTH = 3   # <-- pick a word length

patterns = letter_patterns.pattern_frequencies(LENGTH)
nw = patterns.attrs['n_words']
print(f'{nw:,} words of length {LENGTH}, {len(patterns):,} (letter, pattern) rows')
patterns.sort_values("share", ascending=False).head(50)

640 words of length 3, 119 (letter, pattern) rows


,letter,pattern,n_positions,count,share,freq,prob
81,q,___,0,638,0.996875,0.192460,0.999965
116,z,___,0,632,0.987500,0.192434,0.999832
47,j,___,0,624,0.975000,0.192028,0.997721
102,v,___,0,622,0.971875,0.192199,0.998607
50,k,___,0,621,0.970313,0.191863,0.996865
110,x,___,0,619,0.967187,0.191624,0.995623
28,f,___,0,598,0.934375,0.179900,0.934705
11,c,___,0,591,0.923438,0.188263,0.978159
39,h,___,0,590,0.921875,0.120524,0.626206
106,w,___,0,582,0.909375,0.172822,0.897933


In [17]:
# every position-subset for one letter, most common first
LETTER = 'f'
patterns[patterns.letter == LETTER].reset_index(drop=True)

,letter,pattern,n_positions,count,share,freq,prob
0,f,___,0,598,0.934375,0.179900,0.934705
1,f,f__,1,34,0.053125,0.011694,0.060756
2,f,__f,1,3,0.004687,0.000014,0.000073
3,f,_f_,1,3,0.004687,0.000006,0.000031
4,f,_ff,2,2,0.003125,0.000854,0.004435


In [18]:
# one letter, dropping the 'absent' (___) row — showing all columns
# (count/share unweighted, freq/prob usage-weighted)
patterns[(patterns.letter == LETTER) & (patterns.n_positions > 0)].reset_index(drop=True)

,letter,pattern,n_positions,count,share,freq,prob
0,f,f__,1,34,0.053125,0.011694,0.060756
1,f,__f,1,3,0.004687,0.000014,0.000073
2,f,_f_,1,3,0.004687,0.000006,0.000031
3,f,_ff,2,2,0.003125,0.000854,0.004435


In [20]:
patterns[patterns['n_positions'] > 1].sort_values("share", ascending=False).head(50)

,letter,pattern,n_positions,count,share,freq,prob
25,e,_ee,2,9,0.014063,1.386318e-03,0.007203
74,o,_oo,2,8,0.012500,9.241012e-04,0.004801
79,p,p_p,2,5,0.007812,8.595168e-05,0.000447
26,e,e_e,2,5,0.007812,1.280294e-04,0.000665
97,t,t_t,2,3,0.004687,4.555124e-06,0.000024
19,d,d_d,2,3,0.004687,1.002113e-03,0.005207
8,b,b_b,2,3,0.004687,4.434978e-05,0.000230
58,l,_ll,2,3,0.004687,3.364769e-03,0.017482
63,m,m_m,2,3,0.004687,1.592279e-04,0.000827
20,d,_dd,2,2,0.003125,1.493296e-04,0.000776
